# CertifyTube Viva Notebook - EBM Pipeline

This notebook is a single-flow demo for viva:
1. Setup
2. Data + feature contract check
3. EBM train + evaluate
4. Local inference + native EBM contributors
5. FastAPI endpoint demo (`/engagement/analyze/ebm`)

In [ ]:
# If running in a fresh Colab runtime, uncomment and run:
# !git clone https://github.com/your-username/certifytube_ml_model.git
# %cd certifytube_ml_model

from pathlib import Path
ROOT = Path.cwd()
print('Working directory:', ROOT)
assert (ROOT / 'app').exists(), 'Run this notebook from project root.'

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd

DATA_PATH = 'data/processed/sessions_features.csv'
CONTRACT_PATH = 'verification/engagement/contracts/feature_contract_v1.json'

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
with open(CONTRACT_PATH, 'r', encoding='utf-8') as f:
    contract = json.load(f)

feature_names = contract['features']
print('Feature version:', contract['feature_version'])
print('Total required features:', len(feature_names))

## Train and Evaluate EBM
Set `RUN_TRAINING = False` if artifacts are already prepared and you only want fast demo.

In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    !python verification/engagement/ebm/training/train.py

In [ ]:
!python verification/engagement/ebm/training/evaluate.py

In [ ]:
import json
from pathlib import Path
import pandas as pd

metrics_path = Path('reports/ebm_metrics_test.json')
if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print('EBM test metrics')
    for k, v in metrics.items():
        print(f'{k}: {v}')

sweep_path = Path('reports/ebm_threshold_sweep.csv')
if sweep_path.exists():
    display(pd.read_csv(sweep_path).head())

In [ ]:
from verification.engagement.ebm.inference.predict import predict_engagement_ebm
from verification.engagement.ebm.explain.ebm_explain import compute_local_ebm, top_contributors_ebm

sample_row = df.iloc[0]
feature_payload = {f: float(sample_row[f]) for f in feature_names}

pred = predict_engagement_ebm(feature_payload)
ebm_rows = compute_local_ebm(feature_payload)
neg, pos = top_contributors_ebm(ebm_rows, k=3)

print('Local prediction:', pred)
print('
Top positive contributors:')
for r in pos:
    print(r)
print('
Top negative contributors:')
for r in neg:
    print(r)

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
payload = {
    'session_id': 'viva-ebm-001',
    'feature_version': 'v1.0',
    'features': feature_payload,
}
response = client.post('/engagement/analyze/ebm', json=payload)
print('Status:', response.status_code)
print('Keys:', list(response.json().keys()))
response.json()

## Viva Speaking Points (EBM)
- Why EBM: glass-box model gives auditable term-level effects.
- Comparison angle: XGBoost for performance, EBM for transparency.
- Evidence to show: EBM metrics + contributor explanations from API response.